In [1]:
import string
import re
from pickle import dump
from unicodedata import normalize
from numpy import array

In [ ]:
def load_doc(filename):
    file=open(filename,mode='rt',encoding='utf-8')
    text=file.read()
    file.close()
    return text

def to_pairs(doc):
    lines=doc.strip().split('\n')
    pairs=[line.split('\t') for line in lines]
    return pairs

def clean_pairs(lines):
    cleaned=list()
    re_print=re.compile('[^%s]'%re.escape(string.printable))
    table=str.maketrans('','',string.punctuation)
    for pair in lines:
        clean_pair=list()
        for line in pair:
            
            line = normalize('NFD', line).encode('ascii', 'ignore')
            line = line.decode('utf-8')
            
            line = line.split()
            
            line = [word.lower() for word in line]
            
            line = [word.translate(table) for word in line]
            line = [re_print.sub('', w) for w in line]
            
            line = [word for word in line if word.isalpha()]
            
            clean_pair.append(' '.join(line))
        cleaned.append(clean_pair)
    return array(cleaned)




def save_clean_data(sentences, filename):
    dump(sentences, open(filename, 'wb'))
    print('Saved: %s' % filename)

# load dataset
filename = '/home/ragab/Desktop/TENSOR/deu.txt'
doc = load_doc(filename)

pairs = to_pairs(doc)

clean_pairs = clean_pairs(pairs)
save_clean_data(clean_pairs,'eng-ger.pkl')
for i in range(100):
    print('[%s]=>[%s]'%(clean_pairs[i,0],clean_pairs[i,1]))


Saved: eng-ger.pkl
[go]=>[geh]
[hi]=>[hallo]
[hi]=>[gru gott]
[run]=>[lauf]
[run]=>[lauf]
[wow]=>[potzdonner]
[wow]=>[donnerwetter]
[duck]=>[kopf runter]
[fire]=>[feuer]
[help]=>[hilfe]
[help]=>[zu hulf]
[hide]=>[versteck dich]
[hide]=>[versteckt euch]
[stay]=>[bleib]
[stop]=>[stopp]
[stop]=>[anhalten]
[wait]=>[warte]
[wait]=>[warte]
[begin]=>[fang an]
[do it]=>[mache es]
[do it]=>[tue es]
[go on]=>[mach weiter]
[hello]=>[hallo]
[hello]=>[sers]
[hello]=>[hallo]
[hurry]=>[beeil dich]
[hurry]=>[schnell]
[i hid]=>[ich versteckte mich]
[i hid]=>[ich habe mich versteckt]
[i ran]=>[ich rannte]
[i see]=>[ich verstehe]
[i see]=>[aha]
[i try]=>[ich versuche es]
[i try]=>[ich probiere es]
[i won]=>[ich hab gewonnen]
[i won]=>[ich habe gewonnen]
[i won]=>[ich habe gewonnen]
[oh no]=>[oh nein]
[relax]=>[entspann dich]
[shoot]=>[feuer]
[shoot]=>[schie]
[smile]=>[lacheln]
[sorry]=>[entschuldigung]
[ask me]=>[frag mich]
[ask me]=>[fragt mich]
[ask me]=>[fragen sie mich]
[attack]=>[angriff]
[attack]=>

In [ ]:
from pickle import load
from pickle import dump
from numpy.random import rand
from numpy.random import shuffle


def load_clean_sentences(filename):
    return load(open(filename, 'rb'))


def save_clean_data(sentences, filename):
    dump(sentences, open(filename, 'wb'))
    print('Saved: %s' % filename)


raw_dataset = load_clean_sentences('/home/ragab/Desktop/TENSOR/eng-ger.pkl')


n_sentences = 15000
dataset = raw_dataset[:n_sentences, :]


shuffle(dataset)


train, test = dataset[:12000], dataset[12000:]


save_clean_data(dataset, 'eng-ger-both.pkl')
save_clean_data(train, 'eng-ger-train.pkl')
save_clean_data(test, 'eng-ger-test.pkl')

Saved: eng-ger-both.pkl
Saved: eng-ger-train.pkl
Saved: eng-ger-test.pkl


In [ ]:
from pickle import load
from numpy import array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.utils import plot_model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import RepeatVector
from tensorflow.keras.layers import TimeDistributed
from tensorflow.keras.callbacks import ModelCheckpoint


def load_clean_sentences(filename):
    return load(open(filename, 'rb'))


def create_tokenizer(lines):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(lines)
    return tokenizer


def max_length(lines):
    return max(len(line.split()) for line in lines)


def encode_sequences(tokenizer, length, lines):
    X = tokenizer.texts_to_sequences(lines)
    X = pad_sequences(X, maxlen=length, padding='post')
    return X


def encode_output(sequences, vocab_size):
    ylist = list()
    for sequence in sequences:
        encoded = to_categorical(sequence, num_classes=vocab_size)
        ylist.append(encoded)
    y = array(ylist)
    y = y.reshape(sequences.shape[0], sequences.shape[1], vocab_size)
    return y


def define_model(src_vocab, tar_vocab, src_timesteps, tar_timesteps, n_units):
    model = Sequential()
    model.add(Embedding(src_vocab, n_units, input_length=src_timesteps, mask_zero=True))
    model.add(LSTM(n_units))
    model.add(RepeatVector(tar_timesteps))
    model.add(LSTM(n_units, return_sequences=True))
    model.add(TimeDistributed(Dense(tar_vocab, activation='softmax')))
    return model

# load datasets
dataset = load_clean_sentences('/home/ragab/Desktop/TENSOR/eng-ger-both.pkl')
train = load_clean_sentences('/home/ragab/Desktop/TENSOR/eng-ger-train.pkl')
test = load_clean_sentences('/home/ragab/Desktop/TENSOR/eng-ger-test.pkl')

# prepare english tokenizer
eng_tokenizer = create_tokenizer(dataset[:, 0])
eng_vocab_size = len(eng_tokenizer.word_index) + 1
eng_length = max_length(dataset[:, 0])
print('English Vocabulary Size: %d' % eng_vocab_size)
print('English Max Length: %d' % (eng_length))

# prepare german tokenizer
ger_tokenizer = create_tokenizer(dataset[:, 1])
ger_vocab_size = len(ger_tokenizer.word_index) + 1
ger_length = max_length(dataset[:, 1])
print('German Vocabulary Size: %d' % ger_vocab_size)
print('German Max Length: %d' % (ger_length))

# prepare training data
trainX = encode_sequences(ger_tokenizer, ger_length, train[:, 1])
trainY = encode_sequences(eng_tokenizer, eng_length, train[:, 0])
trainY = encode_output(trainY, eng_vocab_size)

# prepare validation data
testX = encode_sequences(ger_tokenizer, ger_length, test[:, 1])
testY = encode_sequences(eng_tokenizer, eng_length, test[:, 0])
testY = encode_output(testY, eng_vocab_size)

# define model
model = define_model(ger_vocab_size, eng_vocab_size, ger_length, eng_length, 256)
model.compile(optimizer='adam', loss='categorical_crossentropy')

# Build the model with input shape
model.build((None, ger_length))

# summarize defined model
print(model.summary())
plot_model(model, to_file='model.png', show_shapes=True)

# fit model
filename = 'model.h5'
checkpoint = ModelCheckpoint(filename, monitor='val_loss', verbose=1, save_best_only=True, mode='min')
model.fit(trainX, trainY, epochs=30, batch_size=64, validation_data=(testX, testY), callbacks=[checkpoint], verbose=2)

English Vocabulary Size: 2888
English Max Length: 5
German Vocabulary Size: 4597
German Max Length: 10


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 10, 256)        │     1,176,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_1 (RepeatVector)  │ (None, 5, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 5, 256)         │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 5, 2888)        │       742,216 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,969,672 (11.33 MB)

 Trainable params: 2,969,672 (11.33 MB)

...

188/188 - 12s - 64ms/step - loss: 0.7277 - val_loss: 1.9028
Epoch 23/30

Epoch 23: val_loss improved from 1.90277 to 1.90216, saving model to model.h5


188/188 - 12s - 63ms/step - loss: 0.6581 - val_loss: 1.9022
Epoch 24/30

Epoch 24: val_loss improved from 1.90216 to 1.88047, saving model to model.h5


188/188 - 12s - 65ms/step - loss: 0.5962 - val_loss: 1.8805
Epoch 25/30

Epoch 25: val_loss did not improve from 1.88047
188/188 - 12s - 64ms/step - loss: 0.5419 - val_loss: 1.8831
Epoch 26/30

Epoch 26: val_loss improved from 1.88047 to 1.87329, saving model to model.h5


188/188 - 13s - 70ms/step - loss: 0.4888 - val_loss: 1.8733
Epoch 27/30

Epoch 27: val_loss did not improve from 1.87329
188/188 - 12s - 63ms/step - loss: 0.4440 - val_loss: 1.8774
Epoch 28/30

Epoch 28: val_loss did not improve from 1.87329
188/188 - 12s - 65ms/step - loss: 0.4032 - val_loss: 1.8736
Epoch 29/30

Epoch 29: val_loss improved from 1.87329 to 1.87195, saving model to model.h5


188/188 - 13s - 68ms/step - loss: 0.3685 - val_loss: 1.8720
Epoch 30/30

Epoch 30: val_loss did not improve from 1.87195
188/188 - 13s - 69ms/step - loss: 0.3369 - val_loss: 1.8758


In [ ]:
from pickle import load
from numpy import array, argmax
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from nltk.translate.bleu_score import corpus_bleu


def load_clean_sentences(filename):
    return load(open(filename, 'rb'))


def create_tokenizer(lines):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(lines)
    return tokenizer


def max_length(lines):
    return max(len(line.split()) for line in lines)


def encode_sequences(tokenizer, length, lines):
    X = tokenizer.texts_to_sequences(lines)
    X = pad_sequences(X, maxlen=length, padding='post')
    return X


def word_for_id(integer, tokenizer):
    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word
    return None


def predict_sequence(model, tokenizer, source):
    prediction = model.predict(source, verbose=0)[0]
    integers = [argmax(vector) for vector in prediction]
    target = list()
    for i in integers:
        word = word_for_id(i, tokenizer)
        if word is None:
            break
        target.append(word)
    return ' '.join(target)


def evaluate_model(model, tokenizer, sources, raw_dataset):
    actual, predicted = list(), list()
    for i, source in enumerate(sources):
        source = source.reshape((1, source.shape[0]))
        translation = predict_sequence(model, tokenizer, source)


        entry = raw_dataset[i]
        raw_src = entry[1]       # German input is at index 1
        raw_target = entry[0]    # English output is at index 0

        if i < 10:
            print('src=[%s], target=[%s], predicted=[%s]' % (raw_src, raw_target, translation))

        actual.append([raw_target.split()])
        predicted.append(translation.split())

    print('BLEU-1: %f' % corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0)))
    print('BLEU-2: %f' % corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0)))
    print('BLEU-3: %f' % corpus_bleu(actual, predicted, weights=(0.3, 0.3, 0.3, 0)))
    print('BLEU-4: %f' % corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25)))



dataset = load_clean_sentences('/home/ragab/Desktop/TENSOR/eng-ger-both.pkl')
train = load_clean_sentences('/home/ragab/Desktop/TENSOR/eng-ger-train.pkl')
test = load_clean_sentences('/home/ragab/Desktop/TENSOR/eng-ger-test.pkl')



# Prepare English tokenizer
eng_tokenizer = create_tokenizer(dataset[:, 0])
eng_vocab_size = len(eng_tokenizer.word_index) + 1
eng_length = max_length(dataset[:, 0])

# Prepare German tokenizer
ger_tokenizer = create_tokenizer(dataset[:, 1])
ger_vocab_size = len(ger_tokenizer.word_index) + 1
ger_length = max_length(dataset[:, 1])


trainX = encode_sequences(ger_tokenizer, ger_length, train[:, 1])
testX = encode_sequences(ger_tokenizer, ger_length, test[:, 1])

model = load_model('/home/ragab/Desktop/TENSOR/model.h5')

print('Evaluating on Training Set:')
evaluate_model(model, eng_tokenizer, trainX, train)

print('\nEvaluating on Test Set:')
evaluate_model(model, eng_tokenizer, testX, test)

Evaluating on Training Set:
src=[bitte lacheln], target=[please smile], predicted=[please smile]
src=[das ist furchterlich], target=[thats awful], predicted=[thats awful]
src=[vertrau uns einfach], target=[just trust us], predicted=[just trust us]
src=[tom isst brot], target=[tom eats bread], predicted=[tom eats bread]
src=[er riecht ubel], target=[he smells bad], predicted=[he smells bad]
src=[er kann schnell laufen], target=[he can run fast], predicted=[he can run fast]
src=[sie war in sicherheit], target=[she was safe], predicted=[she was safe]
src=[die haben mich gesehen], target=[they saw me], predicted=[they saw me]
src=[was kann ich machen], target=[what can i do], predicted=[what can i do]
src=[das ist in ordnung], target=[thats okay], predicted=[thats ok]
BLEU-1: 0.897836
BLEU-2: 0.854404
BLEU-3: 0.774541
BLEU-4: 0.474079

Evaluating on Test Set:
src=[ich habe verstopfung], target=[im constipated], predicted=[i have homesick]
src=[sprechen sie die wahrheit], target=[tell the t